# 102 — Shadow Mask Tiles

Downloads source tiles for tiles that have lupine predictions,
computes a brightness-based shadow mask per tile, saves to `shadow_tiles/`.

- Only processes tiles with ≥10 lupine pixels (from binary tiles)
- Resume support — skips already-processed tiles
- Visual inspection cell at the bottom for threshold tuning

Run after `101_predict.ipynb`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import time
import gc
import numpy as np

SHDW_DIR = shadow_tile_dir()

print(f"Config loaded ✓")
print(f"  Brightness threshold: {BRIGHTNESS_THRESH}")
print(f"  Shadow tiles dir:     {SHDW_DIR}")
print(f"  Mode:                 {MODE}")
check_required_files()

In [ ]:
# ── Build tile list — only tiles with lupine ──────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

grid = gpd.read_file(GRID_PATH)

if MODE == "region":
    gdf_regions = gpd.read_file(REGION_PATH)
    region_row  = gdf_regions[gdf_regions[REGION_ID_COL] == REGION_ID]
    region_geom = region_row.to_crs(grid.crs).geometry.values[0]
    work_grid   = grid[grid.intersects(region_geom)].copy()
    print(f"  Mode: region {REGION_ID}  |  Tiles: {len(work_grid)}")
else:
    work_grid = grid.copy()
    print(f"  Mode: all tiles  |  Total: {len(work_grid)}")

# Find tiles with >=10 lupine pixels in binary output
bin_dir   = binary_tile_dir(PIXEL_THRESHOLDS[0])
bin_files = [f for f in os.listdir(bin_dir) if f.endswith(".tif")]
print(f"  Scanning {len(bin_files)} binary tiles for lupine …")

lupine_tiles = set()
for i, fname in enumerate(bin_files):
    with rasterio.open(os.path.join(bin_dir, fname)) as src:
        data = src.read(1)
    if int((data == 1).sum()) >= 1:
        source_fname = fname.replace(f"_binary_t{PIXEL_THRESHOLDS[0]}.tif", ".tif")
        lupine_tiles.add(source_fname)
    if (i+1) % 1000 == 0:
        print(f"    [{i+1}/{len(bin_files)}] scanned …")

print(f"  Tiles with >=1 lupine px: {len(lupine_tiles)} / {len(bin_files)}")

# Build url_map filtered to lupine tiles in work_grid
url_map = {
    row["path"].split("/")[-1]: row["path"]
    for _, row in work_grid.iterrows()
    if row["path"].split("/")[-1] in lupine_tiles
}
all_fnames = sorted(url_map.keys())

# Resume
existing  = set(os.listdir(SHDW_DIR))
remaining = [f for f in all_fnames
             if f.replace(".tif", "_shadow.tif") not in existing]

print(f"  Tiles to process: {len(all_fnames)}")
print(f"  Done:             {len(all_fnames) - len(remaining)}")
print(f"  Remaining:        {len(remaining)}")

# Save lupine tile list for use by downstream scripts
tile_list_path = os.path.join(MODEL_OUT_DIR, "lupine_tiles.txt")
with open(tile_list_path, "w") as f:
    for fname in sorted(lupine_tiles):
        f.write(fname + "\n")
print(f"  Tile list saved → {tile_list_path}")

In [ ]:
# ── Download → brightness → save shadow tile ──────────────────────────────────
print("── Processing tiles ────────────────────────────────────────")

skipped = 0
t0      = time.time()

for i, fname in enumerate(remaining):
    url             = url_map[fname]
    local_path      = os.path.join(TMP_DIR, fname)
    shadow_fname    = fname.replace(".tif", "_shadow.tif")
    shadow_path     = os.path.join(SHDW_DIR, shadow_fname)
    remaining_count = len(remaining) - i - 1

    # Download
    if not os.path.exists(local_path):
        ok = safe_download(url, local_path)
        if not ok:
            print(f"  [!] Download failed — skipping {fname}")
            skipped += 1
            continue

    # Read
    try:
        with rasterio.open(local_path) as src:
            img     = src.read().astype(np.float32) / 255.0
            profile = src.profile.copy()
    except Exception as e:
        print(f"  [!] Read failed {fname}: {e}")
        skipped += 1
        try: os.remove(local_path)
        except: pass
        continue

    # Compute brightness shadow mask
    brightness = (img[0] + img[1] + img[2] + img[3]) / 4
    shadow     = (brightness < BRIGHTNESS_THRESH).astype(np.uint8)
    del img, brightness; gc.collect()

    # Save shadow tile
    shadow_profile = profile.copy()
    shadow_profile.update(count=1, dtype="uint8", compress="lzw",
                          tiled=True, blockxsize=256, blockysize=256,
                          nodata=None)
    with rasterio.open(shadow_path, "w", **shadow_profile) as dst:
        dst.write(shadow[np.newaxis, :, :])
    del shadow; gc.collect()

    # Delete source tile
    if DELETE_AFTER_PREDICT:
        try: os.remove(local_path)
        except: pass

    # Progress
    elapsed = time.time() - t0
    done    = i + 1 - skipped
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h")
        print(f"     Saved:   {len(os.listdir(SHDW_DIR))} shadow tiles\n")

print(f"\n  Done. Skipped: {skipped}  |  Time: {(time.time()-t0)/3600:.1f}h")
print(f"  Shadow tiles saved: {len(os.listdir(SHDW_DIR))}")

## Visual inspection — adjust threshold

Run the cell below to check the shadow mask on a specific tile.
Change `tile_fname` and the thresholds list to tune.

In [ ]:
# ── Visual inspection — single tile ───────────────────────────────────────────
import matplotlib.pyplot as plt
import requests as _req

tile_fname   = "933_337_SE_92_31_20250817.tif"
thresholds   = [0.10, 0.15, 0.20]   # compare these

grid_full = gpd.read_file(GRID_PATH)
full_map  = {row["path"].split("/")[-1]: row["path"] for _, row in grid_full.iterrows()}

if tile_fname not in full_map:
    print(f"[!] Tile not found: {tile_fname}")
else:
    tile_path = os.path.join(TMP_DIR, tile_fname)
    if not os.path.exists(tile_path):
        print(f"Downloading {tile_fname} ...")
        r = _req.get(full_map[tile_fname], stream=True, timeout=60)
        with open(tile_path, "wb") as f:
            for chunk in r.iter_content(1024*1024): f.write(chunk)
        print("Done")

    with rasterio.open(tile_path) as src:
        img = src.read().astype(np.float32) / 255.0

    brightness = (img[0] + img[1] + img[2] + img[3]) / 4

    fig, axes = plt.subplots(2, len(thresholds)+1, figsize=(20, 10))

    axes[0, 0].imshow(np.clip(img[:3].transpose(1,2,0), 0, 1))
    axes[0, 0].set_title("RGB"); axes[0, 0].axis("off")
    b_plot = axes[1, 0].imshow(brightness, cmap="gray", vmin=0, vmax=0.4)
    axes[1, 0].set_title("Brightness"); axes[1, 0].axis("off")
    plt.colorbar(b_plot, ax=axes[1, 0])

    for col, thresh in enumerate(thresholds, 1):
        s = (brightness < thresh).astype(np.uint8)
        axes[0, col].imshow(np.clip(img[:3].transpose(1,2,0), 0, 1), alpha=0.7)
        axes[0, col].imshow(s, cmap="Reds", alpha=0.5)
        axes[0, col].set_title(f"t={thresh}  ({s.mean():.1%})")
        axes[0, col].axis("off")
        axes[1, col].hist(brightness.ravel(), bins=80, color="gray", alpha=0.8)
        axes[1, col].axvline(thresh, color="red", linestyle="--", label=f"t={thresh}")
        axes[1, col].set_xlabel("Brightness"); axes[1, col].legend()
        axes[1, col].grid(alpha=0.3)

    plt.suptitle(f"{tile_fname}", fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f"Current BRIGHTNESS_THRESH={BRIGHTNESS_THRESH}: {(brightness < BRIGHTNESS_THRESH).mean():.2%} shadow")